# ML-Based Link Adaptation — Data Generation Notebook

This notebook is **completely self-contained** — no imports from other project files needed.

It includes:
1. The full channel simulator code
2. Data generation
3. Train / val / test split
4. Save to disk
5. Visualisation of the generated data

Just run all cells top to bottom!

## Cell 1 — Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from dataclasses import dataclass, field
from typing import Optional, List
from pathlib import Path

print('All imports OK')

---
## Cell 2 — MCS Table

19 MCS entries spanning QPSK → 256-QAM, based on 5G NR / LTE specifications.

Each row: `(mcs_index, modulation_order, code_rate, spectral_efficiency bps/Hz)`

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MCS TABLE  (index, modulation order, code rate, spectral eff)
# ─────────────────────────────────────────────────────────────────
MCS_TABLE = [
    (0,  2, 0.118, 0.2344),   # QPSK
    (1,  2, 0.188, 0.3770),
    (2,  2, 0.300, 0.6016),
    (3,  2, 0.438, 0.8770),
    (4,  2, 0.588, 1.1758),
    (5,  4, 0.369, 1.4766),   # 16-QAM
    (6,  4, 0.479, 1.9141),
    (7,  4, 0.601, 2.4063),
    (8,  4, 0.754, 3.0156),
    (9,  6, 0.490, 2.9297),   # 64-QAM
    (10, 6, 0.554, 3.3223),
    (11, 6, 0.650, 3.9023),
    (12, 6, 0.754, 4.5234),
    (13, 6, 0.853, 5.1152),
    (14, 8, 0.666, 5.3320),   # 256-QAM
    (15, 8, 0.772, 6.1758),
    (16, 8, 0.873, 6.9844),
    (17, 8, 0.926, 7.4063),
    (18, 8, 0.948, 7.5859),
]

NUM_MCS = len(MCS_TABLE)

# SNR threshold (dB) at which each MCS meets BLER < 10% in AWGN
MCS_SNR_THRESHOLD_AWGN = [
    -6.5, -4.5, -2.0,  0.5,  3.0,
     5.0,  7.0,  9.0, 11.0, 12.5,
    14.0, 15.5, 17.0, 18.5, 20.0,
    21.5, 23.0, 24.5, 26.0,
]

print(f'MCS table loaded: {NUM_MCS} entries')
print(f'Range: QPSK (SE={MCS_TABLE[0][3]}) → 256-QAM (SE={MCS_TABLE[-1][3]}) bps/Hz')

---
## Cell 3 — BLER Model & Throughput Function

**BLER** (Block Error Rate) is modelled with a sigmoid around the SNR threshold:

$$\text{BLER}(\gamma, m) = \frac{1}{1 + e^{k(\gamma - \gamma_m^*)}}$$

**Effective throughput** = spectral efficiency × bandwidth × (1 − BLER)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# BLER MODEL
# ─────────────────────────────────────────────────────────────────
def bler_model(snr_db, mcs_index, channel_type='awgn', slope=2.0):
    """
    Approximate Block Error Rate using a sigmoid around the SNR threshold.

    Parameters
    ----------
    snr_db       : Effective SNR in dB
    mcs_index    : MCS index (0..18)
    channel_type : 'awgn' or 'rayleigh' (adds ~3 dB penalty)
    slope        : Steepness of the waterfall curve

    Returns
    -------
    BLER in [0, 1]
    """
    threshold = MCS_SNR_THRESHOLD_AWGN[mcs_index]
    if channel_type in ('rayleigh', 'jakes'):
        threshold += 3.0          # Rayleigh diversity penalty
    delta = snr_db - threshold
    bler  = 1.0 / (1.0 + np.exp(slope * delta))
    return float(np.clip(bler, 1e-5, 1.0 - 1e-5))


# ─────────────────────────────────────────────────────────────────
# EFFECTIVE THROUGHPUT
# ─────────────────────────────────────────────────────────────────
def effective_throughput(snr_db, mcs_index, channel_type='awgn',
                         bandwidth_hz=20e6):
    """
    Effective throughput = spectral_eff × bandwidth × (1 − BLER)
    Returns throughput in bps.
    """
    _, _, _, spec_eff = MCS_TABLE[mcs_index]
    bler = bler_model(snr_db, mcs_index, channel_type)
    return spec_eff * bandwidth_hz * (1.0 - bler)


# ─────────────────────────────────────────────────────────────────
# SNR → CQI MAPPING
# ─────────────────────────────────────────────────────────────────
def snr_to_cqi(snr_db):
    """Map SNR (dB) to CQI index (1..15) using LTE-like mapping."""
    cqi = int(np.round((snr_db + 10.0) / 2.5))
    return int(np.clip(cqi, 1, 15))


# ─────────────────────────────────────────────────────────────────
# OPTIMAL MCS (oracle / label)
# ─────────────────────────────────────────────────────────────────
def optimal_mcs(snr_db, channel_type='rayleigh', bler_target=0.1):
    """
    Return the highest MCS whose BLER <= bler_target at the given SNR.
    This is the training label (oracle).
    """
    best = 0
    for mcs in range(NUM_MCS):
        if bler_model(snr_db, mcs, channel_type) <= bler_target:
            best = mcs
    return best


# Quick test
print('BLER model test:')
print(f'  SNR=15dB, MCS=10, AWGN    → BLER = {bler_model(15, 10, "awgn"):.4f}')
print(f'  SNR=15dB, MCS=10, Rayleigh → BLER = {bler_model(15, 10, "rayleigh"):.4f}')
print(f'  SNR=15dB, Rayleigh → Optimal MCS = {optimal_mcs(15, "rayleigh")}')
print(f'  SNR=15dB, Rayleigh → Throughput  = {effective_throughput(15, optimal_mcs(15), "rayleigh")/1e6:.2f} Mbps')

---
## Cell 4 — Channel Simulator

Three channel models:
- **AWGN** — constant mean + small Gaussian noise
- **Rayleigh** — log-normal shadowing + Rayleigh fast fading
- **Jakes** — time-correlated Rayleigh fading (Doppler / mobility)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CHANNEL CONFIGURATION
# ─────────────────────────────────────────────────────────────────
@dataclass
class ChannelConfig:
    channel_type   : str   = 'rayleigh'  # 'awgn' | 'rayleigh' | 'jakes'
    snr_mean_db    : float = 15.0        # mean SNR in dB
    snr_std_db     : float = 6.0         # log-normal shadowing std (dB)
    doppler_hz     : float = 50.0        # Doppler freq for Jakes model
    sample_rate_hz : float = 1000.0      # frames per second
    seed           : Optional[int] = 42  # random seed (None = random)


# ─────────────────────────────────────────────────────────────────
# DATASET CONTAINER
# ─────────────────────────────────────────────────────────────────
@dataclass
class ChannelDataset:
    snr_trace    : np.ndarray
    features     : np.ndarray
    labels_mcs   : np.ndarray
    labels_tput  : np.ndarray
    config       : ChannelConfig
    feature_names: List[str] = field(default_factory=list)

    def train_val_test_split(self, val_frac=0.15, test_frac=0.15):
        """Chronological (non-shuffled) train/val/test split."""
        n      = len(self.labels_mcs)
        n_test = int(n * test_frac)
        n_val  = int(n * val_frac)
        idx_train = np.arange(0, n - n_val - n_test)
        idx_val   = np.arange(n - n_val - n_test, n - n_test)
        idx_test  = np.arange(n - n_test, n)
        return idx_train, idx_val, idx_test

    def summary(self):
        lines = [
            f'Channel type   : {self.config.channel_type}',
            f'Mean SNR       : {self.config.snr_mean_db:.1f} dB',
            f'SNR std        : {self.config.snr_std_db:.1f} dB',
            f'Total samples  : {len(self.labels_mcs):,}',
            f'Feature dim    : {self.features.shape[1]}',
            f'MCS range      : [{self.labels_mcs.min()}, {self.labels_mcs.max()}]',
            f'Mean throughput: {self.labels_tput.mean()/1e6:.2f} Mbps',
        ]
        return '\n'.join(lines)


# ─────────────────────────────────────────────────────────────────
# CHANNEL SIMULATOR
# ─────────────────────────────────────────────────────────────────
class ChannelSimulator:
    """
    Simulates time-varying wireless channel and produces per-frame
    SNR, CQI, optimal MCS, and throughput observations.
    """

    def __init__(self, config=None):
        self.cfg = config or ChannelConfig()
        self.rng = np.random.default_rng(self.cfg.seed)
        self._jakes_phase = None

    # ── Jakes correlated fading ───────────────────────────────────
    def _jakes_fading_db(self, n_samples, n_sinusoids=16):
        """
        Generate correlated Rayleigh fading envelope using Jake's model.
        Returns fading power in dB.
        """
        fd = self.cfg.doppler_hz
        fs = self.cfg.sample_rate_hz
        t  = np.arange(n_samples) / fs

        if self._jakes_phase is None:
            self._jakes_phase = self.rng.uniform(0, 2*np.pi, n_sinusoids)

        alpha = (2*np.pi*np.arange(1, n_sinusoids+1)
                 - np.pi + self._jakes_phase) / (4*n_sinusoids)

        # I and Q components
        I = np.sum(
            np.cos(2*np.pi*fd * np.outer(np.cos(alpha), t)), axis=0
        ) / np.sqrt(n_sinusoids)

        Q = np.sum(
            np.sin(2*np.pi*fd * np.outer(np.sin(alpha), t)), axis=0
        ) / np.sqrt(n_sinusoids)

        power_linear = I**2 + Q**2
        return 10 * np.log10(np.clip(power_linear, 1e-10, None))

    # ── SNR trace generation ──────────────────────────────────────
    def generate_snr_trace(self, n_frames):
        """
        Generate n_frames of effective SNR values (dB).

        AWGN    : constant mean + tiny Gaussian noise
        Rayleigh: log-normal shadowing + i.i.d. Rayleigh fast fading
        Jakes   : log-normal shadowing + time-correlated Rayleigh fading
        """
        mean   = self.cfg.snr_mean_db
        std    = self.cfg.snr_std_db

        # Slow fading — log-normal shadowing
        shadow = self.rng.normal(0, std, n_frames)

        if self.cfg.channel_type == 'awgn':
            fast_fading = self.rng.normal(0, 0.5, n_frames)

        elif self.cfg.channel_type == 'rayleigh':
            # Rayleigh envelope from two Gaussian components
            h = (self.rng.normal(0, 1, n_frames)**2 +
                 self.rng.normal(0, 1, n_frames)**2) / 2.0
            fast_fading = 10 * np.log10(np.clip(h, 1e-10, None))

        elif self.cfg.channel_type == 'jakes':
            fast_fading = self._jakes_fading_db(n_frames)

        else:
            raise ValueError(f'Unknown channel type: {self.cfg.channel_type}')

        return mean + shadow + fast_fading

    # ── Full dataset generation ───────────────────────────────────
    def generate_dataset(self, n_frames=100_000, window=8):
        """
        Simulate n_frames of channel observations and build the
        feature matrix + labels for supervised / RL training.

        Feature vector per frame (18 dimensions):
          [0..7]  snr_t-7 .. snr_t   (8-step history window)
          [8]     snr_t               (current SNR)
          [9]     cqi_t               (CQI index)
          [10]    snr_mean_w          (window mean)
          [11]    snr_std_w           (window std)
          [12]    snr_diff1           (1st temporal difference)
          [13]    snr_diff2           (2nd temporal difference)
          [14]    snr_diff3           (3rd temporal difference)
          [15]    ch_awgn             (channel type one-hot)
          [16]    ch_rayleigh
          [17]    ch_jakes

        Labels:
          labels_mcs  : optimal MCS index (0..18)
          labels_tput : effective throughput at optimal MCS (bps)
        """
        print(f'  Generating {n_frames:,} frames of "{self.cfg.channel_type}" channel...')
        snr_trace = self.generate_snr_trace(n_frames)

        # Channel type one-hot
        channel_oh = {
            'awgn':     [1, 0, 0],
            'rayleigh': [0, 1, 0],
            'jakes':    [0, 0, 1],
        }[self.cfg.channel_type]

        features, labels_mcs, labels_tput = [], [], []

        for t in range(window, n_frames):
            win     = snr_trace[t - window : t]   # past 8 SNR values
            snr_now = snr_trace[t]                # current SNR
            cqi_now = snr_to_cqi(snr_now)         # CQI

            # Build feature vector
            feat  = list(win)                               # [0..7] history
            feat += [snr_now]                               # [8]  current SNR
            feat += [cqi_now]                               # [9]  CQI
            feat += [win.mean(), win.std()]                 # [10,11] stats
            feat += [snr_now - win[-1]]                     # [12] 1st diff
            feat += [win[-1] - win[-2], win[-2] - win[-3]] # [13,14] 2nd,3rd diff
            feat += channel_oh                              # [15,16,17] OHE

            # Compute labels
            opt = optimal_mcs(snr_now, self.cfg.channel_type)
            tpt = effective_throughput(snr_now, opt, self.cfg.channel_type)

            features.append(feat)
            labels_mcs.append(opt)
            labels_tput.append(tpt)

        # Feature names
        feat_names = ([f'snr_t-{window-1-i}' for i in range(window)] +
                      ['snr_t', 'cqi_t', 'snr_mean_w', 'snr_std_w',
                       'snr_diff1', 'snr_diff2', 'snr_diff3',
                       'ch_awgn', 'ch_rayleigh', 'ch_jakes'])

        return ChannelDataset(
            snr_trace     = snr_trace,
            features      = np.array(features,   dtype=np.float32),
            labels_mcs    = np.array(labels_mcs, dtype=np.int64),
            labels_tput   = np.array(labels_tput,dtype=np.float32),
            config        = self.cfg,
            feature_names = feat_names,
        )


print('Channel simulator defined successfully!')

---
## Cell 5 — Generate the Dataset

In [ ]:
# ── Configure ─────────────────────────────────────────────────────
cfg = ChannelConfig(
    channel_type   = 'rayleigh',  # change to 'awgn' or 'jakes' to try others
    snr_mean_db    = 15.0,
    snr_std_db     = 6.0,
    seed           = 42,
)

# ── Generate ──────────────────────────────────────────────────────
sim = ChannelSimulator(cfg)
ds  = sim.generate_dataset(n_frames=100_000, window=8)

print('\nDataset Summary:')
print('-' * 40)
print(ds.summary())

---
## Cell 6 — Train / Val / Test Split

In [ ]:
idx_train, idx_val, idx_test = ds.train_val_test_split(
    val_frac  = 0.15,
    test_frac = 0.15,
)

X_train = ds.features[idx_train];   y_train = ds.labels_mcs[idx_train]
X_val   = ds.features[idx_val];     y_val   = ds.labels_mcs[idx_val]
X_test  = ds.features[idx_test];    y_test  = ds.labels_mcs[idx_test]

snr_test = ds.snr_trace[idx_test + 8]   # +8 because of the window offset

print(f'Train : {X_train.shape}  labels: {y_train.shape}')
print(f'Val   : {X_val.shape}  labels: {y_val.shape}')
print(f'Test  : {X_test.shape}  labels: {y_test.shape}')
print(f'\nFeature names:')
for i, name in enumerate(ds.feature_names):
    print(f'  [{i:2d}] {name}')

---
## Cell 7 — Save to Disk

In [ ]:
save_dir = Path('data')
save_dir.mkdir(exist_ok=True)

np.save(save_dir / 'features.npy',    ds.features)
np.save(save_dir / 'labels_mcs.npy',  ds.labels_mcs)
np.save(save_dir / 'labels_tput.npy', ds.labels_tput)
np.save(save_dir / 'snr_trace.npy',   ds.snr_trace)
np.save(save_dir / 'idx_train.npy',   idx_train)
np.save(save_dir / 'idx_val.npy',     idx_val)
np.save(save_dir / 'idx_test.npy',    idx_test)

# Save a text summary too
with open(save_dir / 'dataset_info.txt', 'w') as f:
    f.write(ds.summary())
    f.write('\n\nFeature names:\n')
    for i, name in enumerate(ds.feature_names):
        f.write(f'  [{i:2d}] {name}\n')

print('Files saved to data/:')
for p in sorted(save_dir.glob('*')):
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name:<25} {size_kb:>8.1f} KB')

---
## Cell 8 — Visualise: SNR Trace

In [ ]:
N = 500   # show first 500 frames

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(ds.snr_trace[:N], color='#2F75B6', linewidth=0.9, alpha=0.9)
ax.axhline(cfg.snr_mean_db, color='red', linestyle='--',
           linewidth=1.2, label=f'Mean SNR = {cfg.snr_mean_db} dB')
ax.fill_between(range(N), ds.snr_trace[:N], cfg.snr_mean_db,
                where=(ds.snr_trace[:N] > cfg.snr_mean_db),
                alpha=0.15, color='green', label='Above mean')
ax.fill_between(range(N), ds.snr_trace[:N], cfg.snr_mean_db,
                where=(ds.snr_trace[:N] < cfg.snr_mean_db),
                alpha=0.15, color='red',   label='Below mean')
ax.set_xlabel('Frame Index', fontsize=11)
ax.set_ylabel('SNR (dB)', fontsize=11)
ax.set_title(f'Channel SNR Trace — {cfg.channel_type.upper()} (first {N} frames)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'SNR stats: mean={ds.snr_trace.mean():.2f} dB  '
      f'std={ds.snr_trace.std():.2f} dB  '
      f'min={ds.snr_trace.min():.2f} dB  '
      f'max={ds.snr_trace.max():.2f} dB')

---
## Cell 9 — Visualise: Optimal MCS vs SNR

In [ ]:
N = 500

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True,
                          gridspec_kw={'hspace': 0.08})

# Top: SNR trace
axes[0].plot(ds.snr_trace[:N], color='#2F75B6', linewidth=0.8)
axes[0].set_ylabel('SNR (dB)', fontsize=10)
axes[0].set_title('SNR and Optimal MCS over Time', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Bottom: MCS labels
# Colour by modulation type
mcs_vals = ds.labels_mcs[:N]
colors_mcs = []
for m in mcs_vals:
    if   m <= 4:  colors_mcs.append('#4C72B0')   # QPSK
    elif m <= 8:  colors_mcs.append('#55A868')   # 16-QAM
    elif m <= 13: colors_mcs.append('#C44E52')   # 64-QAM
    else:         colors_mcs.append('#8172B2')   # 256-QAM

axes[1].step(range(N), mcs_vals, where='post',
             color='#333333', linewidth=1.0, alpha=0.7)
axes[1].scatter(range(N), mcs_vals, c=colors_mcs, s=4, zorder=3)
axes[1].set_ylabel('Optimal MCS', fontsize=10)
axes[1].set_xlabel('Frame Index', fontsize=10)
axes[1].set_ylim(-0.5, NUM_MCS - 0.5)
axes[1].grid(True, alpha=0.3)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4C72B0', label='QPSK (0–4)'),
    Patch(facecolor='#55A868', label='16-QAM (5–8)'),
    Patch(facecolor='#C44E52', label='64-QAM (9–13)'),
    Patch(facecolor='#8172B2', label='256-QAM (14–18)'),
]
axes[1].legend(handles=legend_elements, fontsize=8,
               loc='upper right', ncol=2)

plt.tight_layout()
plt.show()

---
## Cell 10 — Visualise: MCS Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# MCS frequency histogram
unique, counts = np.unique(ds.labels_mcs, return_counts=True)
bar_colors = []
for m in unique:
    if   m <= 4:  bar_colors.append('#4C72B0')
    elif m <= 8:  bar_colors.append('#55A868')
    elif m <= 13: bar_colors.append('#C44E52')
    else:         bar_colors.append('#8172B2')

axes[0].bar(unique, counts / len(ds.labels_mcs) * 100,
            color=bar_colors, alpha=0.85, edgecolor='white')
axes[0].set_xlabel('MCS Index', fontsize=10)
axes[0].set_ylabel('Frequency (%)', fontsize=10)
axes[0].set_title('MCS Label Distribution', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

# SNR distribution
axes[1].hist(ds.snr_trace, bins=60, color='#2F75B6',
             alpha=0.8, edgecolor='white', density=True)
axes[1].axvline(ds.snr_trace.mean(), color='red', linestyle='--',
                linewidth=1.5, label=f'Mean = {ds.snr_trace.mean():.1f} dB')
axes[1].set_xlabel('SNR (dB)', fontsize=10)
axes[1].set_ylabel('Density', fontsize=10)
axes[1].set_title('SNR Distribution', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle(f'Dataset Statistics — {cfg.channel_type.upper()} Channel',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print('MCS distribution (train set):')
u, c = np.unique(y_train, return_counts=True)
for mcs, cnt in zip(u, c):
    bar = '█' * (cnt * 40 // len(y_train))
    print(f'  MCS {mcs:2d} | {bar:<40} {cnt/len(y_train)*100:.1f}%')

---
## Cell 11 — Visualise: BLER Waterfall Curves

In [ ]:
snr_range  = np.linspace(-10, 30, 300)
mcs_to_plot = [0, 4, 8, 12, 15, 18]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, ch, title in zip(axes, ['awgn', 'rayleigh'],
                          ['AWGN', 'Rayleigh Fading']):
    cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(mcs_to_plot)))
    for mcs, color in zip(mcs_to_plot, cmap):
        blers = [bler_model(snr, mcs, ch) for snr in snr_range]
        mod   = MCS_TABLE[mcs][1]
        rate  = MCS_TABLE[mcs][2]
        ax.semilogy(snr_range, blers, color=color, linewidth=1.8,
                    label=f'MCS {mcs} ({mod}QAM, R={rate:.2f})')

    ax.axhline(0.1, color='red', linestyle='--',
               linewidth=1.2, label='BLER target (10%)')
    ax.set_xlabel('SNR (dB)', fontsize=10)
    ax.set_ylabel('BLER', fontsize=10)
    ax.set_title(f'BLER Waterfall — {title}', fontsize=11)
    ax.legend(fontsize=7.5, loc='lower left')
    ax.set_ylim(1e-4, 1)
    ax.set_xlim(-10, 30)
    ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

---
## Cell 12 — Reload Saved Data (how to use in training)

Once saved, you can reload in any other script or notebook like this:

In [ ]:
# ── Reload from disk ──────────────────────────────────────────────
features   = np.load('data/features.npy')
labels_mcs = np.load('data/labels_mcs.npy')
labels_tput= np.load('data/labels_tput.npy')
snr_trace  = np.load('data/snr_trace.npy')
idx_train  = np.load('data/idx_train.npy')
idx_val    = np.load('data/idx_val.npy')
idx_test   = np.load('data/idx_test.npy')

# Reconstruct splits
X_train = features[idx_train];   y_train = labels_mcs[idx_train]
X_val   = features[idx_val];     y_val   = labels_mcs[idx_val]
X_test  = features[idx_test];    y_test  = labels_mcs[idx_test]

print('Data reloaded successfully!')
print(f'X_train : {X_train.shape}')
print(f'X_val   : {X_val.shape}')
print(f'X_test  : {X_test.shape}')
print(f'\nSample row (X_test[0]):')
print(f'  {X_test[0]}')
print(f'  → MCS label : {y_test[0]}')
print(f'  → Throughput: {labels_tput[idx_test[0]]/1e6:.2f} Mbps')